In [1]:
from src.graph.graph_pipeline import build_pipeline, run_pipeline

pipe = build_pipeline()
frage = "Wie verändert sich die Luftfeuchtigkeit in Epoche 0 im Durchschnitt?"
antwort = run_pipeline(pipe, frage)

print("Antwort des Systems:")
print(antwort)


[query_processor] input.query:
Wie verändert sich die Luftfeuchtigkeit in Epoche 0 im Durchschnitt?
[query_processor] input.prompt_preview:

                        You are an expert in Neo4j Cypher. Your task is to generate exactly one valid Cypher query based on the user's question.

                        STRICT RULES:
                        1. Only produce a single Cypher query.
                        2. Use only MATCH, OPTIONAL MATCH, WHERE, AND, OR, RETURN, ORDER BY, LIMIT.
                        3. Do not use CREATE, MERGE, DELETE, SET, CALL, LOAD CSV, APOC, or any write operations.
                        4. Do not add explanations, comments, natural language, or code fences.
                        5. Do not wrap the...

[retriever] input.cypher:
MATCH (s:Sensor)-[:GENERATED]->(m:Measurement)-[:EPOCH_IN]->(e:Epoch {epoch_id: 0})
RETURN avg(m.humidity) AS average_humidity

[organizer] input.documents:
documents_total=1

[generator] input.query:
Wie verändert sich die Luftf

In [3]:
import csv
import json
import re
import os
from collections import Counter
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from tqdm import tqdm
from dotenv import load_dotenv

from haystack.components.evaluators import FaithfulnessEvaluator
from haystack.dataclasses import ChatMessage
from haystack_integrations.components.generators.ollama import OllamaChatGenerator

# Load environment variables
load_dotenv()

# -----------------------------
# Config
# -----------------------------
DATASET_PATH = Path("eval_dataset.jsonl")
OUTPUT_DIR = Path("outputs") / "graph_rag_eval"
RUN_NAME = "graphrag_faithfulness"

OLLAMA_URL = os.getenv("OLLAMA_URL", "http://ollama:11434")
# Use the generator model or a specific judge model if defined
JUDGE_MODEL = os.getenv("OLLAMA_GENERATOR_MODEL") or os.getenv("OLLAMA_MODEL", "llama3.1:8b")

MAX_SAMPLES: Optional[int] = None
SHOW_PROGRESS = True

# Debug: store intermediate component outputs into JSON results
SAVE_COMPONENT_OUTPUTS = True
MAX_DEBUG_DOCS = 50

# -----------------------------
# Metrics
# -----------------------------
def _normalize(text: str) -> str:
    text = (text or "").lower().strip()
    text = re.sub(r"[^\w\s]+", " ", text)
    return " ".join(text.split())


def exact_match(pred: str, gold: str) -> bool:
    return _normalize(pred) == _normalize(gold)


def f1_score(pred: str, gold: str) -> float:
    pred_tokens = _normalize(pred).split()
    gold_tokens = _normalize(gold).split()
    if not pred_tokens or not gold_tokens:
        return 0.0

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


# -----------------------------
# IO helpers
# -----------------------------
def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def save_results_json(path: Path, payload: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def save_results_csv(path: Path, rows: List[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    headers = [
        "id",
        "category",
        "question",
        "gold_answer",
        "prediction",
        "exact_match",
        "f1",
        "faithfulness",
        "faithfulness_error",
    ]
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        for r in rows:
            writer.writerow({h: r.get(h, "") for h in headers})


# -----------------------------
# Pipeline + judge
# -----------------------------
def run_pipeline_with_debug(pipe, question: str) -> Tuple[str, str, Dict[str, Any]]:
    """Runs the GraphRAG pipeline and (optionally) captures intermediate outputs."""
    include = ["query_processor", "retriever", "organizer", "generator"]
    result = pipe.run(
        {
            "query_processor": {"query": question},
            "generator": {"query": question},
        },
        include_outputs_from=include,
    )

    answer = result.get("generator", {}).get("answer", "")
    context = result.get("organizer", {}).get("context", "")

    debug: Dict[str, Any] = {
        "query_processor": {
            "cypher": result.get("query_processor", {}).get("cypher"),
        },
        "retriever": {
            "documents_total": None,
            "documents": None,
        },
        "organizer": {
            "context": context,
        },
        "generator": {
            "answer": answer,
        },
    }

    docs = result.get("retriever", {}).get("documents")
    if isinstance(docs, list):
        debug["retriever"]["documents_total"] = len(docs)
        debug["retriever"]["documents"] = [
            {
                "content": getattr(d, "content", None),
                "meta": getattr(d, "meta", None),
            }
            for d in docs[:MAX_DEBUG_DOCS]
        ]

    return answer, context, debug


def _extract_json_object(text: Optional[str]) -> Optional[str]:
    if not text:
        return None
    text = text.strip()

    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            return json.dumps(parsed, ensure_ascii=False)
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None

    candidate = text[start : end + 1]
    try:
        parsed = json.loads(candidate)
        if isinstance(parsed, dict):
            return json.dumps(parsed, ensure_ascii=False)
    except Exception:
        return None

    return None


class JSONCoercingChatGenerator:
    """Small wrapper to keep FaithfulnessEvaluator robust if the judge adds extra text."""

    def __init__(self, base: OllamaChatGenerator):
        self._base = base

    def run(self, messages: List[ChatMessage]):
        res = self._base.run(messages=messages)
        replies = res.get("replies", [])
        if replies and replies[0].text:
            coerced = _extract_json_object(replies[0].text)
            if coerced is not None:
                replies[0] = ChatMessage.from_assistant(coerced)
        return {"replies": replies}


faithfulness_response_schema = {
    "type": "object",
    "properties": {
        "statements": {"type": "array", "items": {"type": "string"}},
        "statement_scores": {"type": "array", "items": {"type": "integer", "enum": [0, 1]}},
    },
    "required": ["statements", "statement_scores"],
}

judge_llm = OllamaChatGenerator(
    model=JUDGE_MODEL,
    url=OLLAMA_URL,
    generation_kwargs={"temperature": 0.0, "num_predict": 256},
    response_format=faithfulness_response_schema,
)
faithfulness = FaithfulnessEvaluator(
    chat_generator=JSONCoercingChatGenerator(judge_llm),
    progress_bar=False,
    raise_on_failure=True,
)


def faithfulness_single(
    *, question: str, context: str, predicted_answer: str, retries: int = 2
) -> Tuple[Optional[float], Optional[dict], Optional[str]]:
    last_err: Optional[Exception] = None
    for _ in range(retries + 1):
        try:
            res = faithfulness.run(
                questions=[question],
                contexts=[[context]],
                predicted_answers=[predicted_answer],
            )
            score = res.get("individual_scores", [None])[0]
            details = res.get("results", [None])[0]
            return score, details, None
        except Exception as exc:
            last_err = exc
    return None, None, str(last_err) if last_err else "unknown error"


# -----------------------------
# Evaluation
# -----------------------------
def evaluate_dataset(
    *,
    pipe,
    dataset_path: Path,
    output_dir: Path,
    run_name: str,
    max_samples: Optional[int] = None,
    show_progress: bool = True,
) -> Dict[str, Any]:
    data = load_jsonl(dataset_path)
    if max_samples is not None:
        data = data[:max_samples]

    results: List[Dict[str, Any]] = []

    iterator = tqdm(data, disable=not show_progress, desc="Evaluating")
    for row in iterator:
        q = str(row.get("question", ""))
        gold = str(row.get("gold_answer", ""))

        component_outputs: Optional[Dict[str, Any]] = None
        try:
            pred, ctx, dbg = run_pipeline_with_debug(pipe, q)
            if SAVE_COMPONENT_OUTPUTS:
                component_outputs = dbg
        except Exception as exc:
            pred, ctx = f"[ERROR] {exc}", ""
            if SAVE_COMPONENT_OUTPUTS:
                component_outputs = {"error": str(exc)}

        em = exact_match(pred, gold)
        f1 = f1_score(pred, gold)

        f_score, f_details, f_err = faithfulness_single(
            question=q,
            context=ctx,
            predicted_answer=pred,
            retries=2,
        )

        result_row: Dict[str, Any] = {
            "id": row.get("id"),
            "category": row.get("category"),
            "question": q,
            "gold_answer": gold,
            "prediction": pred,
            "context": ctx,
            "exact_match": em,
            "f1": f1,
            "faithfulness": f_score,
            "faithfulness_details": f_details,
        }
        if f_err:
            result_row["faithfulness_error"] = f_err
        if SAVE_COMPONENT_OUTPUTS:
            result_row["component_outputs"] = component_outputs

        results.append(result_row)

    valid_faith = [r["faithfulness"] for r in results if isinstance(r.get("faithfulness"), (int, float))]
    avg_faith = float(sum(valid_faith) / len(valid_faith)) if valid_faith else 0.0

    summary = {
        "samples": len(results),
        "exact_match": round(sum(1 for r in results if r.get("exact_match")) / len(results), 4) if results else 0.0,
        "avg_f1": round(sum(float(r.get("f1", 0.0)) for r in results) / len(results), 4) if results else 0.0,
        "faithfulness_score": round(avg_faith, 4),
        "hallucination_rate": round(1.0 - avg_faith, 4),
        "faithfulness_valid_samples": len(valid_faith),
        "judge_model": JUDGE_MODEL,
        "ollama_url": OLLAMA_URL,
        "dataset": str(dataset_path),
        "save_component_outputs": SAVE_COMPONENT_OUTPUTS,
        "max_debug_docs": MAX_DEBUG_DOCS,
    }

    out_json = output_dir / f"{run_name}.json"
    out_csv = output_dir / f"{run_name}.csv"
    save_results_json(out_json, {"summary": summary, "results": results})
    save_results_csv(out_csv, results)

    return {"summary": summary, "json_path": str(out_json), "csv_path": str(out_csv)}


evaluation = evaluate_dataset(
    pipe=pipe,
    dataset_path=DATASET_PATH,
    output_dir=OUTPUT_DIR,
    run_name=RUN_NAME,
    max_samples=MAX_SAMPLES,
    show_progress=SHOW_PROGRESS,
)

print("Summary:")
print(evaluation["summary"])
print("Saved:", evaluation["json_path"], "and", evaluation["csv_path"])

PromptBuilder has 3 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.
Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]


[query_processor] input.query:
Welche Temperatur hat Sensor 1 am 2004-03-31 um 03:38:15.757551 gemessen?
[query_processor] input.prompt_preview:

                        You are an expert in Neo4j Cypher. Your task is to generate exactly one valid Cypher query based on the user's question.

                        STRICT RULES:
                        1. Only produce a single Cypher query.
                        2. Use only MATCH, OPTIONAL MATCH, WHERE, AND, OR, RETURN, ORDER BY, LIMIT.
                        3. Do not use CREATE, MERGE, DELETE, SET, CALL, LOAD CSV, APOC, or any write operations.
                        4. Do not add explanations, comments, natural language, or code fences.
                        5. Do not wrap the...

[retriever] input.cypher:
MATCH (s:Sensor {mote_id: 1})-[:GENERATED]->(m:Measurement)-[:AT_DATE]->(d:Date {date: "2004-03-31"})-[:HAS_HOUR]->(h:Hour {hour: 3})-[:HAS_MINUTE]->(mi:Minute {minute: 38})
RETURN m.temperature AS temperature

[organizer] i

Evaluating:   3%|▎         | 1/30 [00:02<01:09,  2.40s/it]


[query_processor] input.query:
Welche Temperatur hat Sensor 1 am 2004-02-28 um 00:59:16.02785 gemessen?
[query_processor] input.prompt_preview:

                        You are an expert in Neo4j Cypher. Your task is to generate exactly one valid Cypher query based on the user's question.

                        STRICT RULES:
                        1. Only produce a single Cypher query.
                        2. Use only MATCH, OPTIONAL MATCH, WHERE, AND, OR, RETURN, ORDER BY, LIMIT.
                        3. Do not use CREATE, MERGE, DELETE, SET, CALL, LOAD CSV, APOC, or any write operations.
                        4. Do not add explanations, comments, natural language, or code fences.
                        5. Do not wrap the...

[retriever] input.cypher:
MATCH (s:Sensor {mote_id: 1})-[:GENERATED]->(m:Measurement)-[:AT_DATE]->(d:Date {date: "2004-02-28"})-[:HAS_HOUR]->(h:Hour {hour: 0})-[:HAS_MINUTE]->(mi:Minute {minute: 59})
RETURN m.temperature AS temperature

[organizer] in

Evaluating:   7%|▋         | 2/30 [00:04<01:09,  2.49s/it]

...


[query_processor] input.query:
Gibt es Auffälligkeiten in den Messwerten von Sensor 19 über den gesamten Zeitraum?
[query_processor] input.prompt_preview:

                        You are an expert in Neo4j Cypher. Your task is to generate exactly one valid Cypher query based on the user's question.

                        STRICT RULES:
                        1. Only produce a single Cypher query.
                        2. Use only MATCH, OPTIONAL MATCH, WHERE, AND, OR, RETURN, ORDER BY, LIMIT.
                        3. Do not use CREATE, MERGE, DELETE, SET, CALL, LOAD CSV, APOC, or any write operations.
                        4. Do not add explanations, comments, natural language, or code fences.
                        5. Do not wrap the...

[retriever] input.cypher:
MATCH (s:Sensor {mote_id: 19})-[:GENERATED]->(m:Measurement)
OPTIONAL MATCH (m)-[:AT_DATE]->(d:Date)
OPTIONAL MATCH (m)-[:AT_WEEK]->(w:Week)
OPTIONAL MATCH (w)-[:HAS_DATE]->(d2:Date)
OPTIONAL MATCH (m)-[:AT_HOUR]->

Evaluating:  97%|█████████▋| 29/30 [01:28<00:03,  3.67s/it]


[query_processor] input.query:
Gibt es Auffälligkeiten in den Messwerten von Sensor 20 über den gesamten Zeitraum?
[query_processor] input.prompt_preview:

                        You are an expert in Neo4j Cypher. Your task is to generate exactly one valid Cypher query based on the user's question.

                        STRICT RULES:
                        1. Only produce a single Cypher query.
                        2. Use only MATCH, OPTIONAL MATCH, WHERE, AND, OR, RETURN, ORDER BY, LIMIT.
                        3. Do not use CREATE, MERGE, DELETE, SET, CALL, LOAD CSV, APOC, or any write operations.
                        4. Do not add explanations, comments, natural language, or code fences.
                        5. Do not wrap the...

[retriever] input.cypher:
MATCH (s:Sensor {mote_id: 20})-[:GENERATED]->(m:Measurement)
OPTIONAL MATCH (m)-[:AT_DATE]->(d:Date)
OPTIONAL MATCH (m)-[:AT_WEEK]->(w:Week)
OPTIONAL MATCH (w)-[:HAS_DATE]->(d2:Date)
OPTIONAL MATCH (m)-[:AT_HOUR]->

Evaluating: 100%|██████████| 30/30 [01:32<00:00,  3.08s/it]

Summary:
{'samples': 30, 'exact_match': 0.0, 'avg_f1': 0.4908, 'faithfulness_score': 0.5687, 'hallucination_rate': 0.4313, 'faithfulness_valid_samples': 30, 'judge_model': 'llama3.1:8b', 'ollama_url': 'http://ollama:11434', 'dataset': 'eval_dataset.jsonl', 'save_component_outputs': True, 'max_debug_docs': 50}
Saved: outputs/graph_rag_eval/graphrag_faithfulness.json and outputs/graph_rag_eval/graphrag_faithfulness.csv
